# CinnaAI — Complete Cinnamon Disease Model Training
**EfficientNetB0 | MixUp Augmentation | Fine-tuning Pipeline**

> Runtime → Change runtime type → **GPU (T4 or A100)** before running.

| Phase | Description |
|-------|-------------|
| 1–9   | Drive mount, imports, dataset cleaning & deduplication |
| 10–13 | Train/val/test split, tf.data pipelines, class weights |
| 14–17 | Build EfficientNetB0 model, compile, **train 20 epochs** (frozen backbone) |
| 18–22 | Load best checkpoint, selective unfreeze, MixUp, **fine-tune 15 epochs** |
| 23    | Export `cinnamon_multi_part_model.h5` + `class_names.json` |

## Phase 1 — Mount Google Drive

In [ ]:
from google.colab import drive
import os
import shutil

if os.path.exists("/content/drive") and os.path.isdir("/content/drive"):
    print("Removing existing /content/drive directory...")
    shutil.rmtree("/content/drive")

os.makedirs("/content/drive", exist_ok=True)
drive.mount("/content/drive", force_remount=True)

## Phase 2 — Imports and reproducibility seeds

In [ ]:
import csv
import hashlib
import json
import random
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version :", tf.__version__)
print("GPU devices        :", tf.config.list_physical_devices("GPU"))

## Phase 3 — Path configuration

In [ ]:
ZIP_PATH = Path(
    "/content/drive/MyDrive/CinnamonAI/dataset/research_images.zip"
)

LOCAL_ZIP_PATH = Path("/content/research_images.zip")
EXTRACT_PATH   = Path("/content/cinnamon_raw")
CLEAN_PATH     = Path("/content/cinnamon_clean")
SPLIT_PATH     = Path("/content/cinnamon_split")

MODEL_DRIVE_PATH = Path(
    "/content/drive/MyDrive/CinnamonAI/trained_models"
)

MODEL_DRIVE_PATH.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Dataset ZIP not found at:\n{ZIP_PATH}\n"
        "Check the Google Drive folder and filename."
    )

print("Dataset ZIP found:", ZIP_PATH)

## Phase 4 — Copy dataset ZIP to Colab local storage

In [ ]:
if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()

shutil.copy2(ZIP_PATH, LOCAL_ZIP_PATH)

print(
    "ZIP copied to Colab:",
    LOCAL_ZIP_PATH,
    f"({LOCAL_ZIP_PATH.stat().st_size / 1024 ** 2:.2f} MB)",
)

## Phase 5 — Extract ZIP and locate dataset root

In [ ]:
if EXTRACT_PATH.exists():
    shutil.rmtree(EXTRACT_PATH)

EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")

matching_directories = list(EXTRACT_PATH.rglob("leaves_diseases"))

if not matching_directories:
    raise FileNotFoundError(
        "Could not find the 'leaves_diseases' folder inside the extracted ZIP. "
        "Check the ZIP structure."
    )

DATASET_ROOT = matching_directories[0]
print("Dataset root:", DATASET_ROOT)

## Phase 6 — Detect classes and count raw images

In [ ]:
SUPPORTED_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".gif", ".webp",
    ".tif", ".tiff",
}

class_directories = sorted(
    directory
    for directory in DATASET_ROOT.iterdir()
    if directory.is_dir()
)

class_names_from_folders = [directory.name for directory in class_directories]

print("Classes detected:")
for index, class_name in enumerate(class_names_from_folders):
    print(f"  {index}: {class_name}")

print("\nNumber of classes:", len(class_names_from_folders))

original_class_counts = {}

for class_directory in class_directories:
    image_paths = [
        path
        for path in class_directory.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    original_class_counts[class_directory.name] = len(image_paths)

print("\nRaw images per class:\n")
for class_name, count in original_class_counts.items():
    print(f"  {class_name:25s}: {count}")

print("\nTotal raw images:", sum(original_class_counts.values()))

## Phase 7 — Clean dataset

- Correct EXIF rotation (phone camera images)
- Convert all images to RGB JPEG at 95% quality
- Deduplicate using SHA-256 pixel hashes
- Filter corrupted / unreadable files
- Detect cross-class label conflicts

In [ ]:
if CLEAN_PATH.exists():
    shutil.rmtree(CLEAN_PATH)

CLEAN_PATH.mkdir(parents=True, exist_ok=True)

converted_count = 0
corrupted_files = []
duplicate_files = []
label_conflicts = []
seen_hashes     = {}   # pixel_hash -> (class_name, path_str)

for class_directory in class_directories:
    destination_class = CLEAN_PATH / class_directory.name
    destination_class.mkdir(parents=True, exist_ok=True)

    image_paths = [
        path
        for path in class_directory.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    for image_path in image_paths:
        try:
            with Image.open(image_path) as image:
                image       = ImageOps.exif_transpose(image)
                image       = image.convert("RGB")
                image_array = np.asarray(image)
                pixel_hash  = hashlib.sha256(
                    image_array.tobytes() + str(image.size).encode()
                ).hexdigest()

                if pixel_hash in seen_hashes:
                    previous_class, previous_path = seen_hashes[pixel_hash]
                    if previous_class != class_directory.name:
                        label_conflicts.append({
                            "image"         : str(image_path),
                            "current_class" : class_directory.name,
                            "previous_image": previous_path,
                            "previous_class": previous_class,
                        })
                    else:
                        duplicate_files.append(str(image_path))
                    continue

                seen_hashes[pixel_hash] = (class_directory.name, str(image_path))

                destination_path = (
                    destination_class
                    / f"{class_directory.name}_{converted_count:07d}.jpg"
                )
                image.save(destination_path, format="JPEG", quality=95, optimize=True)
                converted_count += 1

        except Exception as error:
            corrupted_files.append({"path": str(image_path), "error": str(error)})

print("Clean images created         :", converted_count)
print("Same-class duplicates skipped:", len(duplicate_files))
print("Corrupted / unreadable files :", len(corrupted_files))
print("Cross-class label conflicts  :", len(label_conflicts))

## Phase 8 — Report conflicts and corrupted files

In [ ]:
if label_conflicts:
    print("\nCross-class label conflicts (first 20):\n")
    for conflict in label_conflicts[:20]:
        print("  Current :", conflict["image"])
        print("  Class   :", conflict["current_class"])
        print("  Clashes :", conflict["previous_image"])
        print("  Class   :", conflict["previous_class"])
        print()
else:
    print("No cross-class label conflicts found.")

if corrupted_files:
    print("\nCorrupted files (first 20):\n")
    for item in corrupted_files[:20]:
        print(f"  {item['path']} -> {item['error']}")
else:
    print("No corrupted files found.")

## Phase 9 — Clean image counts

In [ ]:
clean_class_directories = sorted(
    directory
    for directory in CLEAN_PATH.iterdir()
    if directory.is_dir()
)

clean_class_counts = {}

for class_directory in clean_class_directories:
    count = len(list(class_directory.glob("*.jpg")))
    clean_class_counts[class_directory.name] = count

print("Clean dataset counts:\n")
for class_name, count in clean_class_counts.items():
    print(f"  {class_name:25s}: {count}")

print("\nTotal clean images:", sum(clean_class_counts.values()))

## Phase 10 — Stratified 70 / 15 / 15 train / validation / test split

In [ ]:
if SPLIT_PATH.exists():
    shutil.rmtree(SPLIT_PATH)

for split_name in ["train", "validation", "test"]:
    (SPLIT_PATH / split_name).mkdir(parents=True, exist_ok=True)

MINIMUM_IMAGES = 10
split_summary  = {}

for class_directory in clean_class_directories:
    class_name  = class_directory.name
    image_paths = sorted(class_directory.glob("*.jpg"))

    if len(image_paths) < MINIMUM_IMAGES:
        raise ValueError(
            f"Class '{class_name}' has only {len(image_paths)} images. "
            f"At least {MINIMUM_IMAGES} are required."
        )

    train_paths, temporary_paths = train_test_split(
        image_paths, test_size=0.30, random_state=SEED, shuffle=True,
    )
    validation_paths, test_paths = train_test_split(
        temporary_paths, test_size=0.50, random_state=SEED, shuffle=True,
    )

    split_groups = {
        "train"     : train_paths,
        "validation": validation_paths,
        "test"      : test_paths,
    }

    split_summary[class_name] = {}

    for split_name, paths in split_groups.items():
        dest = SPLIT_PATH / split_name / class_name
        dest.mkdir(parents=True, exist_ok=True)
        for source_path in paths:
            shutil.copy2(source_path, dest / source_path.name)
        split_summary[class_name][split_name] = len(paths)

print("Split summary:\n")
for class_name, counts in split_summary.items():
    print(
        f"  {class_name:25s}  "
        f"Train={counts['train']:4d} | "
        f"Val={counts['validation']:4d} | "
        f"Test={counts['test']:4d}"
    )

## Phase 11 — Load tf.data pipelines

> NOTE: `drop_remainder` is handled inside `apply_mixup` in Phase 20 (since `image_dataset_from_directory` doesn't support it in TF 2.20).

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# First-stage training uses integer labels (SparseCategoricalCrossentropy).
# NOTE: drop_remainder is NOT available in image_dataset_from_directory in TF 2.20.
# The partial-batch fix for MixUp is handled inside apply_mixup() using
# .unbatch().batch(BATCH_SIZE, drop_remainder=True).
train_dataset = tf.keras.utils.image_dataset_from_directory(
    SPLIT_PATH / "train",
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    SPLIT_PATH / "validation",
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    SPLIT_PATH / "test",
    labels="inferred",
    label_mode="int",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names       = train_dataset.class_names
NUMBER_OF_CLASSES = len(class_names)

print("\nTensorFlow class order (alphabetical):")
for index, class_name in enumerate(class_names):
    print(f"  {index}: {class_name}")

print("\nNumber of classes:", NUMBER_OF_CLASSES)

train_dataset      = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset       = test_dataset.prefetch(AUTOTUNE)

print("Datasets prefetched.")

## Phase 12 — Visualise sample images from the training set

In [ ]:
plt.figure(figsize=(12, 10))

for images, labels in train_dataset.take(1):
    for index in range(min(12, len(images))):
        plt.subplot(3, 4, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(class_names[int(labels[index])], fontsize=9)
        plt.axis("off")

plt.suptitle("Training Set — Sample Images", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Phase 13 — Compute class weights (imbalance correction)

In [ ]:
training_counts = []

for class_name in class_names:
    class_directory = SPLIT_PATH / "train" / class_name
    count           = len(list(class_directory.glob("*.jpg")))
    training_counts.append(count)

training_counts       = np.array(training_counts)
total_training_images = int(training_counts.sum())

class_weights = {
    class_index: (
        total_training_images / (NUMBER_OF_CLASSES * class_count)
    )
    for class_index, class_count in enumerate(training_counts)
}

print("Training image counts and class weights:\n")
for class_index, class_name in enumerate(class_names):
    print(
        f"  {class_index}  {class_name:25s}  "
        f"count={training_counts[class_index]:4d}  "
        f"weight={class_weights[class_index]:.4f}"
    )

## Phase 14 — Data augmentation layer

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(mode="horizontal"),
        tf.keras.layers.RandomRotation(factor=0.08),
        tf.keras.layers.RandomZoom(height_factor=0.12, width_factor=0.12),
        tf.keras.layers.RandomTranslation(height_factor=0.08, width_factor=0.08),
        tf.keras.layers.RandomContrast(factor=0.15),
    ],
    name="data_augmentation",
)

print("Data augmentation layer created.")

## Phase 15 — Build EfficientNetB0 model

Backbone is **fully frozen** at this stage. Only the classification head
(GlobalAveragePooling → BatchNorm → Dropout → Dense) trains in Phase 17.

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMAGE_SIZE + (3,),
)

base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name="input_image")
x = data_augmentation(inputs, training=True)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.BatchNormalization(name="classification_batch_norm")(x)
x = tf.keras.layers.Dropout(rate=0.35, name="classification_dropout")(x)

outputs = tf.keras.layers.Dense(
    NUMBER_OF_CLASSES,
    activation="softmax",
    name="disease_predictions",
)(x)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="cinnamon_leaf_condition_model",
)

model.summary()

## Phase 16 — First-stage compile (Adam 1e-3, SparseCategoricalCrossentropy)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=min(3, NUMBER_OF_CLASSES),
            name="top_3_accuracy",
        ),
    ],
)

print("Model compiled — Adam 1e-3 | SparseCategoricalCrossentropy.")

## Phase 17 — Train 20 epochs (classifier head only, backbone frozen)

Callbacks:
- **ModelCheckpoint** — saves best `val_loss` to Drive
- **EarlyStopping** — patience 6, restores best weights
- **ReduceLROnPlateau** — factor 0.3, patience 3, min 1e-7

In [ ]:
BEST_MODEL_PATH = MODEL_DRIVE_PATH / "best_cinnamon_leaf_model.keras"
INITIAL_EPOCHS  = 20

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

print(f"Starting Phase 17 — initial training for {INITIAL_EPOCHS} epochs...")

initial_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

print("\nPhase 17 complete.")

### Phase 17 — Training curves

In [ ]:
history_dict = initial_history.history
epochs_ran   = range(1, len(history_dict["accuracy"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history_dict["accuracy"],     label="Train accuracy")
ax1.plot(epochs_ran, history_dict["val_accuracy"], label="Val accuracy")
ax1.set_title("Accuracy — Initial Training (Phase 17)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, history_dict["loss"],     label="Train loss")
ax2.plot(epochs_ran, history_dict["val_loss"], label="Val loss")
ax2.set_title("Loss — Initial Training (Phase 17)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Phase 18 — Load best checkpoint for fine-tuning

In [ ]:
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"best_cinnamon_leaf_model.keras not found at:\n{BEST_MODEL_PATH}\n"
        "Phase 17 training must complete successfully first."
    )

model = tf.keras.models.load_model(str(BEST_MODEL_PATH), compile=False)

print("best_cinnamon_leaf_model.keras loaded for fine-tuning.")
print("Input  shape:", model.input_shape)
print("Output shape:", model.output_shape)
model.summary(expand_nested=False)

## Phase 19 — Selective backbone unfreeze

| Layer prefix | Action |
|---|---|
| `block7a_*` | **TRAINABLE** |
| `block7b_*` | **TRAINABLE** |
| `top_*` | **TRAINABLE** |
| All others | **FROZEN** |
| **All `BatchNormalization`** | **ALWAYS FROZEN** — preserves ImageNet running statistics |

In [ ]:
base_model = model.get_layer("efficientnetb0")
base_model.trainable = True  # Open globally, then re-freeze selectively.

UNFREEZE_PREFIXES = ("block7a_", "block7b_", "top_")

frozen_count     = 0
trainable_count  = 0
bn_forced_frozen = 0

for layer in base_model.layers:
    is_bn              = isinstance(layer, tf.keras.layers.BatchNormalization)
    starts_with_target = layer.name.startswith(UNFREEZE_PREFIXES)

    if is_bn:
        layer.trainable  = False
        bn_forced_frozen += 1
        frozen_count     += 1
    elif starts_with_target:
        layer.trainable = True
        trainable_count += 1
    else:
        layer.trainable = False
        frozen_count    += 1

print("Selective freeze complete.")
print(f"  Trainable (non-BN top blocks) : {trainable_count}")
print(f"  Frozen layers                 : {frozen_count}")
print(f"  BatchNorm layers force-frozen : {bn_forced_frozen}")
print()

print("Last 30 EfficientNetB0 layers:\n")
for layer in base_model.layers[-30:]:
    status = "TRAIN " if layer.trainable else "FROZEN"
    print(f"  [{status}]  {layer.name:50s}  {type(layer).__name__}")

## Phase 20 — MixUp augmentation tf.data wrapper

MixUp blends two batches element-wise. We use `.unbatch().batch(batch_size, drop_remainder=True)`
to guarantee every batch is exactly `batch_size` images. This prevents the `InvalidArgumentError: Incompatible shapes`
that occurs when mixing a full batch with a partial final batch.

In [ ]:
MIXUP_ALPHA = 0.4

def apply_mixup(
    dataset,
    num_classes,
    batch_size,
    alpha=0.4,
):
    """
    Wrap a batched tf.data.Dataset to apply per-batch MixUp augmentation.

    The function internally calls .unbatch().batch(batch_size, drop_remainder=True)
    to guarantee every batch is exactly batch_size images, preventing shape mismatches.
    """

    def _one_hot_encode(images, labels):
        return images, tf.one_hot(
            tf.cast(labels, tf.int32),
            depth=num_classes,
        )

    def _mixup_batch(batch_a, batch_b):
        images_a, labels_a = batch_a
        images_b, labels_b = batch_b

        g1  = tf.squeeze(tf.random.gamma(shape=(1,), alpha=alpha))
        g2  = tf.squeeze(tf.random.gamma(shape=(1,), alpha=alpha))
        lam = g1 / (g1 + g2 + 1e-8)

        mixed_images = lam * images_a + (1.0 - lam) * images_b
        mixed_labels = lam * labels_a + (1.0 - lam) * labels_b

        return mixed_images, mixed_labels

    one_hot_ds = (
        dataset
        .map(_one_hot_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .unbatch()
        .batch(batch_size, drop_remainder=True)
    )

    one_hot_ds_shuffled = one_hot_ds.shuffle(
        buffer_size=128,
        reshuffle_each_iteration=True,
    )

    return (
        tf.data.Dataset
        .zip((one_hot_ds, one_hot_ds_shuffled))
        .map(_mixup_batch, num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )


mixup_train_dataset = apply_mixup(
    train_dataset,
    num_classes=NUMBER_OF_CLASSES,
    batch_size=BATCH_SIZE,
    alpha=MIXUP_ALPHA,
)

# Validation: clean one-hot labels, no MixUp.
val_dataset_one_hot = validation_dataset.map(
    lambda images, labels: (
        images,
        tf.one_hot(tf.cast(labels, tf.int32), depth=NUMBER_OF_CLASSES),
    ),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)

# Sanity check.
for batch_images, batch_labels in mixup_train_dataset.take(1):
    print("MixUp batch — images:", batch_images.shape, batch_images.dtype)
    print("MixUp batch — labels:", batch_labels.shape, batch_labels.dtype)
    print("Label row 0 sum (== 1.0):", batch_labels[0].numpy().sum())

print(f"\nMixUp pipeline ready  (alpha={MIXUP_ALPHA})")

## Phase 21 — Recompile for fine-tuning (Adam 1e-5, CategoricalCrossentropy)

In [ ]:
FINETUNE_LR = 1e-5

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(
        from_logits=False,
        label_smoothing=0.05,
    ),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=min(3, NUMBER_OF_CLASSES),
            name="top_3_accuracy",
        ),
    ],
)

trainable_params     = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
non_trainable_params = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))

print(f"Compiled — Adam lr={FINETUNE_LR} | CategoricalCrossentropy(label_smoothing=0.05)")
print(f"Trainable parameters     : {trainable_params:,}")
print(f"Non-trainable parameters : {non_trainable_params:,}")

## Phase 22 — Fine-tune for 15 additional epochs

In [ ]:
FINETUNE_EPOCHS     = 15
FINETUNE_MODEL_PATH = MODEL_DRIVE_PATH / "best_cinnamon_leaf_model_finetuned.keras"

finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FINETUNE_MODEL_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        mode="max",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1,
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]

print(f"Starting fine-tuning — {FINETUNE_EPOCHS} epochs | lr={FINETUNE_LR}")
print(f"Best checkpoint -> {FINETUNE_MODEL_PATH}\n")

finetune_history = model.fit(
    mixup_train_dataset,
    validation_data=val_dataset_one_hot,
    epochs=FINETUNE_EPOCHS,
    callbacks=finetune_callbacks,
    verbose=1,
)

print("\nPhase 22 fine-tuning complete.")

### Phase 22 — Fine-tuning curves

In [ ]:
history_dict = finetune_history.history
epochs_ran   = range(1, len(history_dict["accuracy"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history_dict["accuracy"],     label="Train accuracy")
ax1.plot(epochs_ran, history_dict["val_accuracy"], label="Val accuracy")
ax1.set_title("Accuracy — Fine-tuning (Phase 22)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, history_dict["loss"],     label="Train loss")
ax2.plot(epochs_ran, history_dict["val_loss"], label="Val loss")
ax2.set_title("Loss — Fine-tuning (Phase 22)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Phase 23 — Export inference model

**Output files:**
- `cinnamon_multi_part_model.h5` — clean inference graph, no optimizer, no data_augmentation layer
- `class_names.json` — 7-class alphabetical order matching `main.py`

**Input contract:** `float32`, shape `(batch, 224, 224, 3)`, range `[0, 255]`, **NO `/255`**

In [ ]:
H5_OUTPUT_PATH = MODEL_DRIVE_PATH / "cinnamon_multi_part_model.h5"
CN_OUTPUT_PATH = MODEL_DRIVE_PATH / "class_names.json"

_base_model          = model.get_layer("efficientnetb0")
_global_pool         = model.get_layer("global_average_pooling")
_classification_bn   = model.get_layer("classification_batch_norm")
_classification_drop = model.get_layer("classification_dropout")
_disease_output      = model.get_layer("disease_predictions")

infer_input = tf.keras.Input(
    shape=IMAGE_SIZE + (3,), dtype="float32", name="input_image"
)

x = _base_model(infer_input, training=False)
x = _global_pool(x)
x = _classification_bn(x, training=False)
x = _classification_drop(x, training=False)
infer_output = _disease_output(x)

inference_model = tf.keras.Model(
    inputs=infer_input,
    outputs=infer_output,
    name="cinnamon_disease_inference_model",
)

inference_model.summary(expand_nested=False)

### Phase 23 — Save, verify, and evaluate

In [ ]:
# Sanity check.
dummy_rgb_0_255 = np.random.randint(0, 256, size=(1, 224, 224, 3)).astype("float32")
test_pred       = inference_model.predict(dummy_rgb_0_255, verbose=0)

assert test_pred.shape == (1, NUMBER_OF_CLASSES)
assert abs(test_pred[0].sum() - 1.0) < 1e-4
print("Pre-save sanity check PASSED.")

# Save H5.
inference_model.save(str(H5_OUTPUT_PATH), include_optimizer=False, save_format="h5")
h5_size_mb = H5_OUTPUT_PATH.stat().st_size / 1024 ** 2
print(f"\nInference model saved : {H5_OUTPUT_PATH}  ({h5_size_mb:.2f} MB)")

# Save class_names.json.
class_names_export = [
    "healthy_cinnamon",
    "leaf_blight",
    "leaf_miner_attack",
    "leaf_patches_fungal",
    "lower_leaf_gall",
    "non_cinnamon",
    "upper_leaf_gall",
]

assert class_names == class_names_export, (
    f"Class name mismatch!\nDataset: {class_names}\nExport : {class_names_export}"
)

with open(str(CN_OUTPUT_PATH), "w", encoding="utf-8") as f:
    json.dump(class_names_export, f, indent=2)

print(f"class_names.json saved : {CN_OUTPUT_PATH}")

# Round-trip verification.
reloaded      = tf.keras.models.load_model(str(H5_OUTPUT_PATH), compile=False)
reloaded_pred = reloaded.predict(dummy_rgb_0_255, verbose=0)
max_diff      = float(np.max(np.abs(test_pred - reloaded_pred)))

print(f"\nRound-trip max diff : {max_diff:.2e}")
print("PASSED" if max_diff < 1e-5 else "WARNING: check TF version")

# Test-set evaluation.
print("\nEvaluating on held-out test set...")
all_true, all_preds = [], []

for images, labels in test_dataset:
    probs = reloaded.predict(images, verbose=0)
    all_preds.extend(np.argmax(probs, axis=1).tolist())
    all_true.extend(labels.numpy().tolist())

all_true  = np.array(all_true)
all_preds = np.array(all_preds)

print("\nClassification Report:\n")
print(classification_report(all_true, all_preds, target_names=class_names_export))

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names_export).plot(
    ax=ax, colorbar=True, xticks_rotation=45
)
ax.set_title("Confusion Matrix — Test Set (Fine-tuned Model)")
plt.tight_layout()
plt.show()

print("\n" + "=" * 65)
print("PIPELINE COMPLETE")
print("=" * 65)
print(f"  Phase 17 model   : {BEST_MODEL_PATH}")
print(f"  Phase 22 model   : {FINETUNE_MODEL_PATH}")
print(f"  Inference H5     : {H5_OUTPUT_PATH}")
print(f"  class_names.json : {CN_OUTPUT_PATH}")
print("  Input contract   : RGB 224x224 float32 range 0-255 (no /255)")
print(f"  Output           : softmax over {len(class_names_export)} classes")
print("=" * 65)